# Nano Relation Extractor

Trains, quantises and packages the model, one stage per cell.

Runs anywhere. Locally it uses the checkout you started it from; on Kaggle or
Colab it clones the repository and installs what the base image lacks. The next
cell works that out for itself.

**On Kaggle:** set the accelerator to a T4 or P100 and turn internet on in the
settings panel before running.

In [ ]:
import importlib.util, os, pathlib, subprocess, sys

REPO = "https://github.com/apptivitypl/nano-relation-extractor"

if importlib.util.find_spec("nano_re") is None:
    on_kaggle = pathlib.Path("/kaggle").exists()
    local_src = pathlib.Path.cwd().parent / "src"

    if local_src.joinpath("nano_re").is_dir():
        sys.path.insert(0, str(local_src))
    elif on_kaggle:
        for name in ("HF_HOME", "HF_DATASETS_CACHE", "TRANSFORMERS_CACHE"):
            os.environ[name] = "/kaggle/temp/hf"
        pathlib.Path("/kaggle/temp/hf").mkdir(parents=True, exist_ok=True)

        checkout = pathlib.Path("/kaggle/working/nano-relation-extractor")
        if not checkout.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO, str(checkout)], check=True
            )
        missing = [
            name
            for name in ("onnxruntime", "onnxscript")
            if importlib.util.find_spec(name) is None
        ]
        if missing:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", *missing], check=True
            )
        sys.path.insert(0, str(checkout / "src"))
        os.chdir(checkout)
    else:
        raise SystemExit(
            "nano_re is not importable. Run 'uv sync' in the project root, "
            "then start Jupyter with 'uv run jupyter lab'."
        )

import torch

import nano_re

print("nano_re", nano_re.__version__)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print(f"gpu {properties.name} {properties.total_memory / 1e9:.0f} GB")

## Choose the scale

`LIMIT` caps documents per corpus, and only that many are downloaded, so it
governs disk, network and time together.

| `LIMIT` | Download | 3 epochs, M4 Pro | 3 epochs, T4 |
| --- | --- | --- | --- |
| 5000 | ~150 MB | ~45 min | ~40 min |
| 20000 | ~600 MB | ~3 h | ~2.5 h |
| 60000 | ~1.8 GB | ~9 h | ~7 h |

Kaggle sessions stop at twelve hours, so leave headroom for the stages after
training. Everything else configures itself from the detected device.

In [ ]:
import os
from dataclasses import replace

from nano_re.config import PipelineConfig
from nano_re.pipeline import Pipeline

LANGUAGES = "pl,en,de,fr,es,it,nl,pt"
LIMIT = 5000
EPOCHS = 3

os.environ.setdefault("NANO_RE_LANGUAGES", LANGUAGES)
os.environ.setdefault("NANO_RE_MAX_SEQUENCE_LENGTH", "384")

config = PipelineConfig.from_env()
config = config.with_overrides(
    data=replace(config.data, limit=LIMIT),
    training=replace(config.training, epochs=EPOCHS),
)
pipeline = Pipeline(config)

print("encoder: ", config.model.backbone_name)
print("languages:", ", ".join(config.data.languages))
print("artifacts:", config.artifacts_dir.resolve())

## Data

Downloads the corpora, interleaves them by weight and derives the label schema
from what was actually read. The relation inventory depends on which languages
are in scope, so it is counted rather than declared.

In [ ]:
schema = pipeline.prepare()

print("Entity types:", ", ".join(schema.entity_types))
print("BIO tags:    ", schema.num_bio_labels)
print("Relations:   ", schema.num_relation_labels)

counts = pipeline.data_module.inventory.counts
top = sorted(counts.items(), key=lambda item: -item[1])[:10]
for relation_id, count in top:
    print(f"  {relation_id:<8} {count:>6}  {schema.describe_relation(relation_id)}")

In [ ]:
bundle = pipeline.data_module.build_corpus(config.data.train_split, training=True)
print(bundle.describe())

for index in range(len(bundle.dataset)):
    sample = bundle.dataset[index]
    if sample is None:
        continue
    print(f"\nFirst usable document: {sample.doc_id}")
    print(f"  sub-words {sample.input_ids.shape[0]}, entities {sample.num_entities},"
          f" candidate pairs {sample.num_pairs}")
    print(f"  relation supervision: {sample.has_relation_supervision}")
    print(f"  mention mask rows sum to one: "
          f"{[round(float(x), 3) for x in sample.mention_mask.sum(-1)[:4]]}")
    break

## Training

Both heads share one encoder and are trained under
`L = alpha * L_NER + beta * L_RE`. A corpus that annotates entities but not
relations is masked out of the relation term.

In [ ]:
training_report = pipeline.train()

best = training_report.best_evaluation
print(f"Best epoch:          {training_report.best_epoch}")
print(f"NER micro F1:        {best.ner.f1:.4f}")
print(f"Relation micro F1:   {best.relation.f1:.4f}")
print(f"Recall ceiling:      {best.relation_recall_ceiling:.4f}")

## Export and quantisation

The exporter compares the graph against PyTorch on three differently shaped
batches and fails if the relative deviation exceeds tolerance or if the two
would ever choose different classes.

In [ ]:
artifacts = pipeline.export()

print("Backend:            ", artifacts.export.exporter)
print("Dynamic shapes:     ", artifacts.export.dynamic_shapes_verified)
print(f"Relative deviation:  {artifacts.export.max_relative_deviation:.2e}")
print("Decisions match:    ", artifacts.export.decisions_match)
print(f"Size: {artifacts.quantization.source_bytes / 1e6:.1f} MB -> "
      f"{artifacts.quantization.target_bytes / 1e6:.1f} MB "
      f"({artifacts.quantization.compression_ratio:.2f}x)")

In [ ]:
benchmark = pipeline.benchmark(measure_accuracy=True)

print(f"FP32: {benchmark.fp32.median_ms:6.2f} ms/page  {benchmark.fp32.size_mb:7.1f} MB")
print(f"INT8: {benchmark.int8.median_ms:6.2f} ms/page  {benchmark.int8.size_mb:7.1f} MB")
print(f"Speedup {benchmark.speedup:.2f}x, size reduction {benchmark.size_reduction:.1%}")
print(f"F1 change from quantisation: NER {benchmark.ner_f1_delta:+.4f}, "
      f"relation {benchmark.relation_f1_delta:+.4f}")

## Bundle

Writes the model card from the measurements above, inventories the directory and
checks that nothing expected is missing.

In [ ]:
report = pipeline.package(
    training=training_report,
    benchmark=benchmark,
    quantization=artifacts.quantization,
)
print(report.render())
print("\nComplete:", report.is_complete)

## Using the model

Text of any length works: it is split into overlapping windows and the results
are merged. Structured identifiers are matched by rule and verified by checksum,
alongside whatever the model predicts.

In [ ]:
from nano_re.inference import RelationExtractor

extractor = RelationExtractor.from_bundle(
    pipeline.artifacts_dir, backend="onnx-int8", config=config
)
print("Backend:", extractor.backend_name, "\n")

text = (
    "Skai TV is a Greek free-to-air television network based in Piraeus. "
    "Skai TV is part of Skai Group, one of the largest media groups in the country."
)
print(extractor.extract(text).render())

In [ ]:
print((pipeline.artifacts_dir / "MODEL_CARD.md").read_text(encoding="utf-8"))

## Trimming the bundle

The float32 graph exists only for the benchmark comparison. Remove it if you are
near a storage limit, such as Kaggle's output quota.

In [ ]:
import pathlib

graph = pipeline.artifacts_dir / "model.onnx"
if graph.exists():
    print(f"removing the {graph.stat().st_size / 1e6:.0f} MB float32 graph")
    graph.unlink()

total = sum(p.stat().st_size for p in pipeline.artifacts_dir.rglob("*") if p.is_file())
print(f"bundle: {total / 1e6:.0f} MB")